# Notebook 1 — Preprocessing Pipeline
## ACL SRW 2026: The Language of AI Liability

Pipeline:
  1. Load corpus
  2. Identify definition sections
  3. Sentence segmentation (spaCy en_core_web_trf)
  4. Deontic modal detection (spaCy Matcher, priority-ordered)
  5. Obligation sentence classification
  6. Export structured dataset for downstream notebooks
  7. Same pipeline for the non-AI baselines and Colorado SB 26-189

In [ ]:
# Run once per environment (Colab or local)
import sys
%pip install -q spacy spacy-curated-transformers
!{sys.executable} -m spacy download en_core_web_trf

In [2]:
# ── Cell 1: Setup ──

import json
import re
import spacy
from spacy.matcher import Matcher
from pathlib import Path
from collections import defaultdict

BASE_DIR = Path("Data")
OUTPUT_DIR = BASE_DIR / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

# Corpus manifest — maps law_id to filename
# Note: EU AI Act uses the binding-only version (recitals excluded)
CORPUS = {
    "eu_ai_act":       "eu_ai_act_binding.txt",
    "china_genai":     "china_genai_measures.txt",
    "south_korea":     "south_korea_ai_basic_act.txt",
    "ca_sb1047":       "ca_sb1047.txt",
    "ca_sb53":         "ca_sb53.txt",
    "ca_sb942":        "ca_sb942.txt",
    "ca_ab2013":       "ca_ab2013.txt",
    "co_sb205":        "co_sb205.txt",
    "tx_traiga":       "tx_traiga.txt",
    "il_hb3773":       "il_hb3773.txt",
}

METADATA = {
    "eu_ai_act":   {"name": "EU AI Act (Reg. 2024/1689)", "jurisdiction": "EU",
                    "year": 2024, "status": "In force (phased)",
                    "source_type": "official_english",
                    "note": "Recitals excluded; Articles + Annexes only"},
    "china_genai": {"name": "Interim Measures for GenAI Services", "jurisdiction": "China",
                    "year": 2023, "status": "In force",
                    "source_type": "unofficial_translation"},
    "south_korea": {"name": "AI Basic Act", "jurisdiction": "South Korea",
                    "year": 2024, "status": "In force (Jan 2026)",
                    "source_type": "partial_official"},
    "ca_sb1047":   {"name": "SB 1047", "jurisdiction": "California",
                    "year": 2024, "status": "Vetoed",
                    "source_type": "official_english"},
    "ca_sb53":     {"name": "SB 53 (TFAIA)", "jurisdiction": "California",
                    "year": 2025, "status": "In force (Jan 2026)",
                    "source_type": "official_english"},
    "ca_sb942":    {"name": "SB 942 (AI Transparency Act)", "jurisdiction": "California",
                    "year": 2024, "status": "In force",
                    "source_type": "official_english"},
    "ca_ab2013":   {"name": "AB 2013", "jurisdiction": "California",
                    "year": 2024, "status": "In force",
                    "source_type": "official_english"},
    "co_sb205":    {"name": "Colorado SB 24-205", "jurisdiction": "Colorado",
                    "year": 2024, "status": "Repealed May 2026 (replaced by SB 26-189)",
                    "source_type": "official_english"},
    "tx_traiga":   {"name": "Texas TRAIGA (HB 149)", "jurisdiction": "Texas",
                    "year": 2025, "status": "In force (Jan 2026)",
                    "source_type": "official_english"},
    "il_hb3773":   {"name": "Illinois HB 3773", "jurisdiction": "Illinois",
                    "year": 2024, "status": "In force (Jan 2026)",
                    "source_type": "official_english"},
}

In [3]:
# ── Fix 1: Illinois backtick cleanup ──

il_path = BASE_DIR / "il_hb3773.txt"
il_text = il_path.read_text(encoding="utf-8")

# Count backticks before fix
backtick_count = il_text.count('`')
print(f"Illinois — backticks before fix: {backtick_count}")

# Strip all backticks
il_fixed = il_text.replace('`', '')

# Also collapse runs of blank lines into single blank line
il_fixed = re.sub(r'\n{3,}', '\n\n', il_fixed)

# Save
il_path.write_text(il_fixed, encoding="utf-8")
print(f"Illinois — backticks after fix: {il_fixed.count('`')}")
print(f"Illinois — words: {len(il_fixed.split()):,}")

# Show a sample of the cleaned text
lines = [l for l in il_fixed.split('\n') if l.strip()]
print(f"\nFirst 10 non-empty lines:")
for line in lines[:10]:
    print(f"  {line.strip()[:120]}")

Illinois — backticks before fix: 0
Illinois — backticks after fix: 0
Illinois — words: 3,887

First 10 non-empty lines:
  **Public Act 103-0804**
  HB3773 Enrolled
  LRB103 27562 SPS 53938 b
  AN ACT concerning business.
  **Be it enacted by the People of the State of Illinois,**
  **represented in the General Assembly:**
  Section 5. The Illinois Human Rights Act is amended by
  changing Sections 2-101 and 2-102 as follows:
  (775 ILCS 5/2-101)
  Sec. 2-101. Definitions. The following definitions are


In [4]:
import spacy
# ── Cell 2: Load spaCy model ──

spacy.prefer_gpu()  # uses a GPU when one is available
nlp = spacy.load("en_core_web_trf")

# Increase max length for the EU AI Act
nlp.max_length = 700_000

print(f"Model loaded: {nlp.meta['name']} ({nlp.meta['version']})")
print(f"Pipeline components: {nlp.pipe_names}")

Model loaded: core_web_trf (3.8.0)
Pipeline components: ['transformer', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [5]:
# ── Cell 3: Definition section detection ──

# Patterns that identify definition sections or definitional sentences.
# Two layers:
#   (a) Section-level: headers like "Definitions" or "As used in this chapter"
#   (b) Sentence-level: patterns like '"X" means' or 'shall mean'

DEFINITION_HEADER_PATTERNS = re.compile(
    r'\b(?:definitions?)\b'
    r'|\bas\s+used\s+in\s+this\s+'
    r'|\bfor\s+(?:the\s+)?purposes?\s+of\s+this\s+',
    re.IGNORECASE
)

DEFINITIONAL_SENTENCE_PATTERNS = re.compile(
    r'\bshall\s+(?:mean|have\s+the\s+(?:same\s+)?meaning)'
    r'|\bmeans?\b.*(?:following|any\s+of\s+the\s+following)'
    r'|["\u201c][^"\u201d]+["\u201d]\s+means\b'
    r'|\bhas\s+the\s+(?:same\s+)?meaning'
    r'|\bis\s+defined\s+(?:in|as)\b',
    re.IGNORECASE
)

def is_definitional_sentence(sent_text):
    """Check if a sentence is definitional (defines a term rather than
    imposing an obligation). Used to filter out 'shall mean' from
    obligation counts."""
    return bool(DEFINITIONAL_SENTENCE_PATTERNS.search(sent_text))


# ── Cell 4: Deontic modal matcher setup ──

def build_deontic_matcher(vocab):
    """Build a spaCy Matcher with priority-ordered deontic modal patterns.

    Pattern groups are added in order of specificity (longest first).
    Overlapping matches are resolved post-hoc by keeping the longest span.

    Categories:
      - PROHIBITION: shall not, must not, may not, is/are prohibited (from),
                     no person/entity shall
      - HARD_OBLIGATION: shall, must, is/are required to
      - SOFT_OBLIGATION: should, is/are expected/encouraged to
      - PERMISSION: may, is/are authorized/permitted to
    """
    matcher = Matcher(vocab)

    # -- PROHIBITION (longest patterns first) --
    matcher.add("PROHIBITION", [
        [{"LOWER": "no"}, {"LOWER": {"IN": ["person", "entity", "one"]}},
         {"LOWER": "shall"}],
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "prohibited"},
         {"LOWER": "from"}],
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "prohibited"}],
        [{"LOWER": "shall"}, {"LOWER": "not"}],
        [{"LOWER": "must"}, {"LOWER": "not"}],
        [{"LOWER": "may"}, {"LOWER": "not"}],
    ])

    # -- HARD OBLIGATION --
    matcher.add("HARD_OBLIGATION", [
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "required"},
         {"LOWER": "to"}],
        [{"LOWER": "shall"}],
        [{"LOWER": "must"}],
    ])

    # -- SOFT OBLIGATION --
    matcher.add("SOFT_OBLIGATION", [
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "expected"},
         {"LOWER": "to"}],
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "encouraged"},
         {"LOWER": "to"}],
        [{"LOWER": "should"}],
    ])

    # -- PERMISSION --
    matcher.add("PERMISSION", [
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "authorized"},
         {"LOWER": "to"}],
        [{"LOWER": {"IN": ["is", "are"]}}, {"LOWER": "permitted"},
         {"LOWER": "to"}],
        [{"LOWER": "may"}],
    ])

    return matcher


def resolve_matches(matches, vocab):
    """Resolve overlapping matches by keeping the longest span at each
    position. Returns a list of (category, start, end) tuples with no
    overlapping token ranges."""
    if not matches:
        return []

    # Convert to (category_name, start, end, span_length)
    decoded = []
    for match_id, start, end in matches:
        cat = vocab.strings[match_id]
        decoded.append((cat, start, end, end - start))

    # Sort by start position, then by span length descending
    decoded.sort(key=lambda x: (x[1], -x[3]))

    resolved = []
    covered = set()
    for cat, start, end, length in decoded:
        token_range = set(range(start, end))
        if token_range & covered:
            continue  # Skip: overlaps with a previously accepted match
        covered |= token_range
        resolved.append((cat, start, end))

    return resolved

In [6]:
# ── Cell 5: Process a single law ──

def process_law(law_id, text, nlp, matcher, min_tokens=4):
    """Process a single law through the full pipeline.

    Returns a dict with metadata, corpus-level stats, and per-sentence
    annotations including deontic classifications.
    """
    doc = nlp(text)
    sentences = []

    for sent_idx, sent in enumerate(doc.sents):
        sent_text = sent.text.strip()
        # Skip empty sentences and segmentation artifacts ("5.", "SEC", "(a)")
        if not sent_text or len(sent) < min_tokens:
            continue

        # Definitional sentence check
        is_definitional = is_definitional_sentence(sent_text)

        # Run deontic matcher on this sentence's span
        sent_doc = sent.as_doc()
        raw_matches = matcher(sent_doc)
        resolved = resolve_matches(raw_matches, nlp.vocab)

        # Extract matched categories and spans
        deontic_matches = []
        for cat, start, end in resolved:
            span_text = sent_doc[start:end].text
            deontic_matches.append({
                "category": cat,
                "span": span_text,
                "start_token": start,
                "end_token": end,
            })

        categories = sorted(set(m["category"] for m in deontic_matches))

        # A sentence is an obligation sentence if:
        # (a) it has at least one deontic match, AND
        # (b) it is not a definitional sentence
        is_obligation = len(deontic_matches) > 0 and not is_definitional

        sentences.append({
            "sent_id": sent_idx,
            "text": sent_text,
            "start_char": sent.start_char,
            "end_char": sent.end_char,
            "token_count": len(sent),
            "is_definitional": is_definitional,
            "deontic_matches": deontic_matches,
            "deontic_categories": categories,
            "is_obligation": is_obligation,
        })

    # Compute corpus-level stats
    total_sents = len(sentences)
    obligation_sents = sum(1 for s in sentences if s["is_obligation"])
    definitional_sents = sum(1 for s in sentences if s["is_definitional"])

    cat_counts = defaultdict(int)
    for s in sentences:
        if s["is_obligation"]:
            for m in s["deontic_matches"]:
                cat_counts[m["category"]] += 1

    return {
        "law_id": law_id,
        "metadata": METADATA[law_id],
        "stats": {
            "total_sentences": total_sents,
            "obligation_sentences": obligation_sents,
            "definitional_sentences": definitional_sents,
            "word_count": len(text.split()),
            "deontic_counts": dict(cat_counts),
        },
        "sentences": sentences,
    }

In [7]:
# ── Cell 6: Run the pipeline on the AI-law corpus and export ──

matcher = build_deontic_matcher(nlp.vocab)
results = {}
for law_id, filename in CORPUS.items():
    text = (BASE_DIR / filename).read_text(encoding="utf-8")
    print(f"Processing {law_id} ({len(text.split()):,} words)...", end=" ")
    results[law_id] = process_law(law_id, text, nlp, matcher, min_tokens=4)
    s = results[law_id]["stats"]
    print(f"done. {s['total_sentences']} sentences, {s['obligation_sentences']} obligation, "
          f"{s['definitional_sentences']} definitional")
# ── Summary table ──

print(f"\n{'Law':<22} {'Words':>7} {'Sents':>6} {'Oblig':>6} "
      f"{'Defn':>5} {'HARD':>5} {'SOFT':>5} {'PERM':>5} {'PROH':>5}")
print("-" * 82)

for law_id in CORPUS:
    r = results[law_id]
    s = r["stats"]
    d = s["deontic_counts"]
    print(f"{law_id:<22} {s['word_count']:>7,} {s['total_sentences']:>6} "
          f"{s['obligation_sentences']:>6} {s['definitional_sentences']:>5} "
          f"{d.get('HARD_OBLIGATION', 0):>5} {d.get('SOFT_OBLIGATION', 0):>5} "
          f"{d.get('PERMISSION', 0):>5} {d.get('PROHIBITION', 0):>5}")

# ── Export ──

output_path = OUTPUT_DIR / "preprocessed_corpus.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nFull results saved to: {output_path}")

summary = {}
for law_id, r in results.items():
    summary[law_id] = {
        "metadata": r["metadata"],
        "stats": r["stats"],
        "obligation_sent_ids": [s["sent_id"] for s in r["sentences"]
                                 if s["is_obligation"]],
    }

summary_path = OUTPUT_DIR / "corpus_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Summary saved to: {summary_path}")

Processing eu_ai_act (55,327 words)... 

done. 833 sentences, 710 obligation, 1 definitional
Processing china_genai (1,991 words)... 

done. 38 sentences, 16 obligation, 0 definitional
Processing south_korea (10,166 words)... 

done. 146 sentences, 102 obligation, 2 definitional
Processing ca_sb1047 (8,223 words)... 

done. 121 sentences, 56 obligation, 9 definitional
Processing ca_sb53 (5,881 words)... 

...

done. 30 sentences, 13 obligation, 8 definitional
Processing ca_ab2013 (829 words)... 

done. 18 sentences, 3 obligation, 6 definitional
Processing co_sb205 (6,106 words)... 

done. 66 sentences, 25 obligation, 11 definitional
Processing tx_traiga (5,706 words)... 

done. 75 sentences, 43 obligation, 13 definitional
Processing il_hb3773 (3,887 words)... 

done. 93 sentences, 16 obligation, 14 definitional

Law                      Words  Sents  Oblig  Defn  HARD  SOFT  PERM  PROH
----------------------------------------------------------------------------------
eu_ai_act               55,327    833    710     1   811     5   164    50
china_genai              1,991     38     16     0    22     0     0    10
south_korea             10,166    146    102     2   119     0    74     4
ca_sb1047                8,223    121     56     9    64     0    17     8
ca_sb53                  5,881     86     54    12    55     1    16    13
ca_sb942                 1,199     30     13     8    12     0     3     1
ca_ab2013                  829     18      3     6     1     0     2     1
co_sb205                 6,106     66     25    11    28     1    16     0
tx_traiga                5,706     75     43    13    34     0    24    11
il_hb3773                3,887     93     16    14     7     0    10     0

Full results saved to: Data/processed/p

In [8]:
# ── Cell 7: Non-AI baselines and Colorado SB 26-189 (same pipeline) ──
# One recent non-AI technology law per jurisdiction, plus the successor of
# Colorado SB 24-205. They stay outside the main corpus.

BASELINES = {
    "eu_dsa":      "eu_dsa.txt",
    "cn_pipl":     "cn_pipl.txt",
    "kr_pipa":     "kr_pipa.txt",
    "co_sb21_190": "co_sb21_190.txt",
    "co_sb26_189": "co_sb26_189.txt",
}
METADATA.update({
    "eu_dsa":      {"name": "Digital Services Act (Reg. 2022/2065)", "jurisdiction": "EU",
                    "year": 2022, "status": "In force", "source_type": "official_english",
                    "note": "Recitals excluded; Articles only", "role": "non-AI baseline"},
    "cn_pipl":     {"name": "Personal Information Protection Law", "jurisdiction": "China",
                    "year": 2021, "status": "In force", "source_type": "unofficial_translation",
                    "role": "non-AI baseline"},
    "kr_pipa":     {"name": "Personal Information Protection Act (Act No. 19234)",
                    "jurisdiction": "South Korea", "year": 2023, "status": "In force",
                    "source_type": "official_translation", "role": "non-AI baseline"},
    "co_sb21_190": {"name": "Colorado Privacy Act (SB 21-190)", "jurisdiction": "Colorado",
                    "year": 2021, "status": "In force", "source_type": "official_english",
                    "role": "non-AI baseline"},
    "co_sb26_189": {"name": "Colorado SB 26-189 (ADMT)", "jurisdiction": "Colorado",
                    "year": 2026, "status": "Signed May 2026; applies Jan 2027",
                    "source_type": "official_english", "role": "successor of SB 24-205"},
})

baseline_results = {}
for law_id, filename in BASELINES.items():
    text = (BASE_DIR / filename).read_text(encoding="utf-8")
    baseline_results[law_id] = process_law(law_id, text, nlp, matcher, min_tokens=4)
    s = baseline_results[law_id]["stats"]
    print(f"{law_id:<12} {s['word_count']:>6,} words  {s['total_sentences']:>4} sentences  "
          f"{s['obligation_sentences']:>4} obligation  {s['definitional_sentences']:>3} definitional")

with open(OUTPUT_DIR / "baselines_corpus.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {OUTPUT_DIR / 'baselines_corpus.json'}")


eu_dsa       33,281 words   619 sentences   552 obligation    2 definitional


cn_pipl       5,733 words   131 sentences    78 obligation    0 definitional


kr_pipa      29,786 words   507 sentences   419 obligation    7 definitional


co_sb21_190   8,421 words   145 sentences    65 obligation   10 definitional


co_sb26_189   5,706 words    90 sentences    35 obligation    7 definitional

Saved: Data/processed/baselines_corpus.json


In [9]:
"""
Spot check: sample obligation and non-obligation sentences
from a few key laws to verify classification quality.
"""

import random
random.seed(42)

SPOT_CHECK_LAWS = ["eu_ai_act", "ca_sb1047", "china_genai", "tx_traiga", "il_hb3773"]

for law_id in SPOT_CHECK_LAWS:
    sents = results[law_id]["sentences"]
    obligation = [s for s in sents if s["is_obligation"]]
    non_obligation = [s for s in sents if not s["is_obligation"] and not s["is_definitional"]]
    definitional = [s for s in sents if s["is_definitional"]]

    print(f"\n{'=' * 70}")
    print(f"{law_id} — {len(obligation)} obligation / "
          f"{len(non_obligation)} non-obligation / {len(definitional)} definitional")
    print(f"{'=' * 70}")

    # Sample 3 obligation sentences
    print(f"\n-- OBLIGATION (sample 3) --")
    for s in random.sample(obligation, min(3, len(obligation))):
        cats = ", ".join(s["deontic_categories"])
        spans = ", ".join(m["span"] for m in s["deontic_matches"])
        text = s["text"][:200] + ("..." if len(s["text"]) > 200 else "")
        print(f"  [{cats}] [{spans}]")
        print(f"  {text}\n")

    # Sample 2 non-obligation sentences
    print(f"-- NON-OBLIGATION (sample 2) --")
    for s in random.sample(non_obligation, min(2, len(non_obligation))):
        text = s["text"][:200] + ("..." if len(s["text"]) > 200 else "")
        print(f"  {text}\n")

    # Show all definitional sentences (usually few)
    if definitional:
        print(f"-- DEFINITIONAL (all {len(definitional)}) --")
        for s in definitional:
            spans = ", ".join(m["span"] for m in s["deontic_matches"])
            text = s["text"][:200] + ("..." if len(s["text"]) > 200 else "")
            print(f"  [matched: {spans}]")
            print(f"  {text}\n")


eu_ai_act — 710 obligation / 122 non-obligation / 1 definitional

-- OBLIGATION (sample 3) --
  [HARD_OBLIGATION] [shall]
  On the basis of the findings, that report shall, where appropriate, be accompanied by a proposal for amendment of this Regulation.

  [PROHIBITION] [shall not]
  This paragraph shall not apply to third parties making accessible to the public tools, services, processes, or components, other than general-purpose AI models, under a free and open-source licence.

  [HARD_OBLIGATION] [shall, is required to]
  CHAPTER III

HIGH-RISK AI SYSTEMS

SECTION 1

Classification of AI systems as high-risk

Article 6

Classification rules for high-risk AI systems

1.   Irrespective of whether an AI system is placed o...

-- NON-OBLIGATION (sample 2) --
  [(35)](#ntc35-L_202401689EN.000101-E0035)  Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the Member States relating to the ...

  If the notifying auth